# Day 1 - Structure & Diagnostics

> Two lab blocks, each running after its deck. Work down in order.
> Cells marked `TODO` are yours to fill in; a `checks.check_ex_*`
> call tells you whether it worked. Stretch sections are optional.

## Setup

Everything the labs need lives in `coursekit`, and the cell below installs it.
On Colab that is the whole setup. Running locally instead? If the cell fails,
`python scripts/check_env.py` from the repo root reports what is missing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/NaifMersal/time-series-analysis-and-forecasting.git /content/ts-course
    %cd /content/ts-course
    !pip install -q -e .

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from coursekit import checks
from coursekit import datasets as D
from coursekit import plotting as P

P.use_course_style()
print("ready")

---
# Before Day 1

*About twenty minutes, on your own, before the first session.* Nothing here is
assessed and none of it is part of a timed lab block. It refreshes the pandas
the course leans on and puts the five ideas of the
[statistics refresher](../slides/00_statistics_refresher.html) through your own
hands. If a section is already obvious to you, skim it.

The setup cell above is all the setup there is: on Colab it clones and installs
the course, and locally it just imports it.

---
## Dates in pandas

A time series is a value **plus a timestamp**, and almost every real-world bug in
forecasting is a timestamp bug. Three things to be fluent in.

In [ ]:
import numpy as np
import pandas as pd

# A DatetimeIndex is not a list of strings.
dates = pd.date_range("2024-01-01", periods=6, freq="MS")   # MS = month START
print(dates)
print("\nyear:", dates.year.tolist())
print("month:", dates.month.tolist())

`freq="MS"` is month-start, `"ME"` is month-end, `"D"` daily, `"h"` hourly,
`"QS"` quarter-start. Getting this wrong shifts your whole series by a month.

In [ ]:
# resample: change the frequency. asfreq: assert one, exposing gaps as NaN.
daily = pd.Series(np.arange(60.0), index=pd.date_range("2024-01-01", periods=60, freq="D"))
print("daily -> monthly totals:")
print(daily.resample("MS").sum())

# Now punch a hole in the calendar and make it visible.
holed = daily.drop(daily.index[10:14])
print(f"\nrows after dropping 4 days: {len(holed)}")
print(f"rows after asfreq('D'):     {len(holed.asfreq('D'))}  "
      f"<- the gap is now VISIBLE as NaN")

**Why this matters.** Dropping four rows does not shift the index - but it *does* mean
row `t-7` is no longer "one week ago". Every seasonal lag after the gap is wrong, and
nothing raises an error. Day 1 comes back to this at the end of the first hour.

In [ ]:
# TODO: `s` below is missing two months. Reindex it onto a complete monthly
#       calendar so the gaps become visible NaNs, then count them.
s = pd.Series(
    [10.0, 11, 13, 12, 15, 16],
    index=pd.to_datetime(["2024-01-01", "2024-02-01", "2024-05-01",
                          "2024-06-01", "2024-07-01", "2024-08-01"]),
)

repaired = ...

assert repaired is not Ellipsis, "Fill in `repaired` above."
print(repaired)
print(f"missing months: {int(repaired.isna().sum())}")

---
## Long format, and groupby

The whole course uses the layout fpppy uses: one row per series per timestamp, with
columns **`unique_id` / `ds` / `y`**. It looks redundant for one series and pays off the
moment you have 148.

In [ ]:
allr = D.retail_all()
print(allr.head())
print(f"\n{allr['unique_id'].nunique()} series, {len(allr):,} rows")

# One number per series - the pattern used constantly on Day 1.
summary = (allr.groupby("unique_id")["y"]
               .agg(n="size", mean="mean", last="last")
               .sort_values("mean", ascending=False))
print("\nBiggest five by average turnover:")
print(summary.head())

---
## The statistics refresher

Five short checks, one per idea the refresher deck builds, on the same real
height measurements the deck is drawn on. Run them in order; each `checks.check_ex_0_*`
call tells you whether it worked. These are the ideas Day 1 and Day 2 assume,
so getting them right here saves time later.

In [ ]:
# Exercise 0.1 - An 80% range is a pair of cuts
#
# A range is not a formula, it is two positions in a sorted list. Take the
# refresher's height survey and cut off the shortest and tallest 10%.
from coursekit import checks

heights = D.height_survey()["height_cm"]

lo = ...   # the 0.1 quantile of `heights`
hi = ...   # the 0.9 quantile of `heights`

checks.check_ex_0_1(heights, lo, hi)

In [ ]:
# Exercise 0.2 - Independence: two strangers, or two brothers
#
# Independence says one value tells you nothing about another. Galton's 1886
# study recorded which family each man belonged to, so the same men can be
# paired two ways. Correlate each pair both times.
strangers = D.unrelated_pairs()   # two men from different families
brothers = D.brother_pairs()      # two men from the same family

r_strangers = ...   # correlation of `strangers["a"]` with `strangers["b"]`
r_brothers = ...    # the same for `brothers`

checks.check_ex_0_2(r_strangers, r_brothers)

In [ ]:
# Exercise 0.3 - Test one claim against two samples
#
# Take the claim that heights follow a bell centred on 175 cm with an SD of 7.
# Cut that bell into ten slices of equal share, count each sample into them,
# and run the chi-squared test. The survey's men are indistinguishable from
# the claim; the same survey with its women added back in is not.
from scipy import stats

MU, SD, K = D.HEIGHT_MEAN, D.HEIGHT_SD, 10
obs_men, expected, edges = P.bell_bin_counts(
    D.height_survey()["height_cm"], MU, SD, K)
obs_pooled, _, _ = P.bell_bin_counts(
    D.height_survey_pooled()["height_cm"], MU, SD, K)

p_men = ...      # the p-value from stats.chisquare(obs_men)
p_pooled = ...   # the same for obs_pooled

checks.check_ex_0_3(p_men, p_pooled)

In [ ]:
# Exercise 0.4 - Detectable is not large
#
# Now test the men against a claim of 173.5 cm, which is 1.8 cm below what they
# actually measure. Do it on all 799 of them, then on the first 50. The gap is
# identical both times, so anything that changes is about how many were counted.
men = D.height_survey()["height_cm"].to_numpy()

p_799 = ...   # chi-squared p-value for all of `men` against 173.5 and SD
p_50 = ...    # the same for its first 50

checks.check_ex_0_4(p_799, p_50)

In [ ]:
# Exercise 0.5 - Build the null distribution yourself
#
# The deck read the critical value and the p-value off a chi-squared curve.
# You do not have to take that curve on faith. Simulate the claim being true:
# draw 5,000 surveys of 799 men from the claimed bell, bin each one and
# compute its Q. Both numbers then fall out of counting.
rng = np.random.default_rng(0)
draws = rng.normal(MU, SD, size=(5000, len(men)))
slots = np.searchsorted(edges, draws)                    # slice each man lands in
counts = np.apply_along_axis(np.bincount, 1, slots, minlength=K)
q_sim = ((counts - expected) ** 2 / expected).sum(axis=1)   # one Q per survey

q_men = float(stats.chisquare(obs_men).statistic)        # the real men's Q

q95 = ...          # the 95th percentile of q_sim; the table says 16.92
p_counted = ...    # the share of q_sim at or past q_men; the table says 0.94

checks.check_ex_0_5(q95, p_counted)

Bring one time series from your own work if you have one - the last exercise on
Day 1 is easy to point at your own data.

---
# Lab A - Structure and patterns

Runs after the first deck. Exercises 1.1 and 1.2, 25 minutes.

---
# Exercise 1.1 - Name the pattern

*10 minutes. No modelling - look and argue.*

Six series are plotted below. For each one decide:

- Is there a **trend**?
- Is there **seasonality** (a *fixed, known* period)?
- Is there a **cycle** (rises and falls at *no* fixed period)?
- Is the seasonal swing **additive** (constant size) or **multiplicative**
  (grows with the level)?

In [ ]:
series = {
    "spine": D.spine(),
    "beer": D.beer(),
    "lynx": D.lynx(),
    "noise": D.white_noise(n=300, seed=7),
    "souvenirs": D.souvenirs(),
    "canadian_gas": D.canadian_gas(),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, df) in zip(axes.ravel(), series.items()):
    P.plot_series(df, ax=ax, title=name)
P.thin_xticks(axes, n=3)
# lynx is annual: decade labels and a tick on every year, so you can count the
# gap between one peak and the next.
P.year_xticks(axes[0, 2], step=10, minor=1, rotation=45)
plt.show()

Fill in your calls below. Use any of: `"trend"`, `"seasonality"`, `"cycle"`,
`"none"`, `"additive"`, `"multiplicative"`.

In [ ]:
answers = {
    "spine":        [],   # TODO
    "beer":         [],   # TODO
    "lynx":         [],   # TODO
    "noise":        [],   # TODO
    "souvenirs":    [],   # TODO
    "canadian_gas": [],   # TODO
}

checks.check_ex_1_1(answers)

> **Discussion.** `lynx` is the one that catches people. Its peaks are 8-11
> years apart - never the same gap twice - so it is a *cycle*, not seasonality.
> Nothing in the calendar produces it.
>
> `canadian_gas` is worth a second look too. Its swing grows with the level, so
> multiplicative is the right call, but its seasonal *shape* also drifts across
> the decades. A fixed additive/multiplicative split cannot express that, which
> is exactly what STL shows you in exercise 1.4.

---
# Exercise 1.2 - Load, verify, and look

*15 minutes.*

The spine for this whole course is Victorian takeaway-food turnover: monthly,
1982-2018.

**Before plotting anything, verify the timestamps.** A silent gap shifts every
seasonal lag after it, and nothing downstream will warn you.

In [ ]:
spine = ...   # TODO: load the spine (see coursekit/datasets.py)
spine.head()

In [ ]:
# TODO: fill in the four checks below.
inferred_freq = ...          # what frequency does pandas infer?
n_duplicates  = ...          # duplicated timestamps
n_expected    = ...          # how many months SHOULD lie between first and last?
n_actual      = ...          # how many rows do we have?

print(f"inferred frequency : {inferred_freq}")
print(f"duplicate stamps   : {n_duplicates}")
print(f"expected / actual  : {n_expected} / {n_actual}")

checks.check_ex_1_2(spine)

This series is clean. Most are not - so here is what a gap actually costs.
Run this and watch the calendar slip.

In [ ]:
# Drop three months at random and see what happens to the calendar.
rng = np.random.default_rng(0)
holes = rng.choice(np.arange(100, 300), size=3, replace=False)
broken = spine.drop(index=holes).reset_index(drop=True)

step = broken["ds"].diff().dt.days
print(f"rows: {len(spine)} -> {len(broken)}, and pandas still reports no error")
print(f"month-to-month steps seen  : {sorted(step.dropna().unique().astype(int).tolist())}")
print(f"steps longer than a month  : {int((step > 32).sum())}")

# "12 rows back" is no longer "12 months back" once a gap is inside the window.
g = int(step.idxmax())          # first row after the biggest gap
i = g + 6                       # a row whose previous 12 span that gap
print(f"\nbiggest jump: {broken['ds'][g - 1].date()} -> {broken['ds'][g].date()}")
print(f"row {i} is {broken['ds'][i].date()}, and 12 rows back is "
      f"{broken['ds'][i - 12].date()}")
print("which is no longer the same month one year earlier")

**Repairing a gap.** Reindex onto the full date range, then decide what the
missing values mean - interpolate, carry forward, or leave `NaN` and use a
model that tolerates them. Never let the gap stay *invisible*.

In [ ]:
full_index = pd.date_range(broken["ds"].min(), broken["ds"].max(), freq="MS")
repaired = (broken.set_index("ds")
                  .reindex(full_index)
                  .rename_axis("ds")
                  .reset_index())
repaired["unique_id"] = repaired["unique_id"].ffill()
print(f"rows: {len(broken)} -> {len(repaired)},  NaNs now visible: {repaired['y'].isna().sum()}")
repaired["y"] = repaired["y"].interpolate()
print(f"after interpolation, NaNs: {repaired['y'].isna().sum()}")

Now the three plots. Each answers a different question.

In [ ]:
sp = D.add_calendar(spine)

# TODO: 1. a time plot of the whole series
# TODO: 2. a seasonal plot (one line per year, month on the x axis)
# TODO: 3. a subseries plot (one panel per month)

**Write your answer here.** In two or three sentences: what is going on in this
series? Mention the trend, the seasonal shape, whether the swing is growing, and
anything unusual around 2009.

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* Turnover rises roughly sixfold in level from 1982 to 2018, with a
clear December peak and February trough every year. The seasonal swing grows
with the level, so the series is multiplicative - the log transform in 1.4
stabilises it. Growth jumps sharply in 2009-2010 and then plateaus
through about 2014. That is a level shift rather than a seasonal one: the
seasonal plot shows the *shape* stays put while the level moves under it.

### Stretch

Pick a second retail series with `D.retail_all()` and contrast it with the
spine. Is its seasonal shape the same? Does it peak in December too?

In [ ]:
# Stretch - your code here.

---
# Lab B - Measure what you saw

Runs after the second deck. Exercises 1.3 to 1.5, 45 minutes.

---
# Exercise 1.3 - Read the correlogram

*15 minutes.*

First, the matching game. Four correlograms below, in a scrambled order.
Match each to its series.

In [ ]:
mystery = {
    "A": D.white_noise(n=400, seed=11),
    "B": D.spine(),
    "C": D.beer(),
    "D": D.lynx(),
}
order = ["C", "A", "D", "B"]      # the plots are drawn in THIS order

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, key in zip(axes, order):
    P.acf_plot(mystery[key]["y"], nlags=30, ax=ax, title=f"correlogram {order.index(key) + 1}")
plt.show()

Which correlogram belongs to which series? Say *why* - name the feature you
used (slow decay, spikes at a period, everything inside the bounds).

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.*

1. **beer** - spikes at lags 4, 8, 12 (quarterly data, m = 4) with little decay.
2. **white noise** - every spike inside the bounds.
3. **lynx** - a slow *wave*: positive at short lags, negative around lag 5,
   positive again near lag 10. A cycle shows as an oscillating ACF, not as
   spikes at a fixed multiple.
4. **spine** - slow decay from near 1.0, the signature of a strong trend, with
   a seasonal ripple riding on top.

In [ ]:
noise = D.white_noise(n=len(spine), seed=7)

# TODO: compute the ACF of the spine and of the noise series, to 36 lags,
#       plus the significance bound.
acf_spine, bound = ...
acf_noise, _ = ...

print(f"bound = {bound:.4f}")
print(f"spine  r_1 = {acf_spine[0]:.3f}   r_12 = {acf_spine[11]:.3f}")
print(f"noise: fraction of lags outside the bounds = "
      f"{(np.abs(acf_noise) > bound).mean():.1%}")

checks.check_ex_1_3(acf_spine, acf_noise, bound)

**Question.** Roughly 5% of white-noise autocorrelations land outside the
bounds *by construction*. If you plot 36 lags, how many spikes outside the band
should stop worrying you?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* About 36 x 0.05 = 1.8, so one or two stray spikes are exactly what
white noise looks like. Treat the bounds as a null hypothesis about a *single*
lag, not as a per-plot decision rule. What matters is a pattern - a run of
spikes, or a spike at a meaningful lag like 12.

---
# Exercise 1.4 - Transform and decompose

*15 minutes.*

The spine's seasonal swing grows with its level. Stabilise it with the log
transform from Deck 1, then split it into trend, seasonal and remainder.

In [ ]:
from statsmodels.tsa.seasonal import STL

# TODO: log-transform the spine, then plot original vs log side by side.
yt = ...

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
P.plot_series(spine, ax=axes[0], title="original")
axes[1].plot(spine["ds"], yt, color=P.ORANGE, lw=0.9)
axes[1].set_title("log scale - the swing is now constant")
plt.show()

In [ ]:
# TODO: run STL on the log-transformed series (remember: you must supply `period`)
#       and assemble a frame with columns ds / transformed / trend / seasonal / remainder.
res = ...
dcmp = ...

fig, axes = P.decomposition_plot(
    dcmp, ["transformed", "trend", "seasonal", "remainder"], "STL")
plt.show()

checks.check_ex_1_4(dcmp)

In [ ]:
# TODO: plot the seasonally adjusted series against the transformed series.

### Stretch

Run a **classical** decomposition (`seasonal_decompose`) on the same series and
compare it with STL. Look hard at the first and last six months.

In [ ]:
# Stretch - your code here.

---
# Exercise 1.5 - Features across a portfolio

*15 minutes.*

One series is a plot. A hundred and forty-eight series need **numbers**.

Compute strength of trend and strength of seasonality for every Australian
retail series, then use them to find the interesting ones.

In [ ]:
def stl_features(g):
    """Return trend and seasonal strength for one series (long format)."""
    y = np.log(np.clip(g["y"].to_numpy(), 1e-6, None))
    r = STL(y, period=12, robust=True).fit()
    R, T_, S = np.asarray(r.resid), np.asarray(r.trend), np.asarray(r.seasonal)
    var_r = np.var(R)
    # TODO: implement the two formulas from the slides.
    trend_strength = ...
    seasonal_strength = ...
    return pd.Series({"trend_strength": trend_strength,
                      "seasonal_strength": seasonal_strength})


allr = D.retail_all()
feat = (allr.groupby("unique_id")[["y"]]
            .apply(stl_features, include_groups=False)
            .reset_index())

checks.check_ex_1_5(feat)
feat.head()

In [ ]:
# TODO: which series are the most and least seasonal? Print the top 5 and bottom 5.

**Now check the numbers meant what you think.** Plot the most and the least
seasonal series side by side. If the feature is doing its job, the difference
should be obvious to the eye.

In [ ]:
# TODO: plot the most and least seasonal series.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.scatter(feat["trend_strength"], feat["seasonal_strength"], s=26,
           color=P.BLUE, alpha=0.6)
mine = feat[feat["unique_id"] == D.SPINE_ID].iloc[0]
ax.scatter([mine["trend_strength"]], [mine["seasonal_strength"]], s=120,
           color=P.BLACK, zorder=3, label="our spine")
ax.set(xlabel="strength of trend", ylabel="strength of seasonality",
       title="148 retail series in feature space",
       xlim=(0, 1.05), ylim=(0, 1.05))
ax.legend(frameon=False)
plt.show()

**Question.** Every one of the 148 series scores above 0.89 on trend strength,
and 97% score above 0.96. Is that feature useless here?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* Useless for *discriminating between these series* - yes. But it is
still a finding: it says "everything in Australian retail trends", which tells
you that any model you pick must handle a trend, and that seasonality is the
axis worth routing on. A feature that does not vary across your portfolio is a
feature you can stop computing - after you have looked at it once.

### Stretch

`stl_features` takes any long-format frame with `unique_id` / `ds` / `y`. Point it
at a series of your own, or at a second series from `D.retail_all()`, and see
where it lands on the map above.

In [ ]:
# TODO: build a one-series frame and run stl_features on it, then say where it
# sits relative to the cloud: more or less seasonal than the spine?

---
## End of Day 1

You can now diagnose a series: see its patterns, measure them with the ACF,
split it into components, and summarise a whole portfolio.

Tomorrow you forecast - and, more importantly, learn how to tell whether the
forecast was any good.